##Setup

Configure the training seed and the training output directory.

In [ ]:
REPO_URL = "https://github.com/humanoid-purdue/legged_rl.git"
OUT_DIR  = "training"
SEED     = 0

In [ ]:
#Clone the repo and uv sync
import os, subprocess, sys
from pathlib import Path

# Headless Mujoco
os.environ["MUJOCO_GL"] = "egl"

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
if not Path("/content/legged_rl").exists():
    subprocess.run(["git", "clone", REPO_URL, "/content/legged_rl"], check=True)
os.chdir("/content/legged_rl")
subprocess.run(["uv", "sync", "--extra", "gpu"], check=True)

In [ ]:
# Check if GPU is available
subprocess.run(["uv", "run", "python", "-c",
                "import jax; print(jax.devices()); "
                "assert jax.default_backend() == 'gpu', 'no GPU runtime'"], check=True)

##Train

Runs the training script with support for checkpoint restoration, then plots the reward curve when training is done.

In [ ]:
# Use the "--restore" argument to run from a checkpoint
# Use the "--num_timesteps" argument to change the number of environment steps when restarting from a checkpoint

subprocess.run(["uv", "run", "python", "train.py", "--seed", str(SEED), "--out_dir", OUT_DIR ], check=True)

In [ ]:
# Plot reward curve
import json
import matplotlib.pyplot as plt

with open(f"{OUT_DIR}/run.json", "r") as file:
  run_data = json.load(file)

num_steps = run_data["eval_curve"]["num_steps"]
episode_reward = run_data["eval_curve"]["episode_reward"]
episode_reward_std = run_data["eval_curve"]["episode_reward_std"]

plt.figure(figsize=(10, 5))

plt.errorbar(
    num_steps,
    episode_reward,
    yerr=episode_reward_std,
    marker="o",
    capsize=3,
)

plt.xlabel("Environment Steps")
plt.ylabel("Mean Episode Reward")
plt.title("Reward Curve")
plt.grid(True)
plt.savefig(f"{OUT_DIR}/reward_curve.png", dpi=140)
plt.show()

##Export to Drive

Copies the full training output directory to a new folder in drive.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

!cp -r "/content/legged_rl/{OUT_DIR}" /content/drive/MyDrive/